<a href="https://colab.research.google.com/github/takatakamanbou/AdvML/blob/2025/AdvML2025_ex04notebookA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AdvML ex04notebookA

<img width=72 src="https://www-tlab.math.ryukoku.ac.jp/~takataka/course/AdvML/AdvML-logo.png"> [この授業のウェブページ](https://www-tlab.math.ryukoku.ac.jp/wiki/?AdvML)




今回の話は，学部の科目「機械学習I」でも出てきています．受講しなかった方や復習したい方は以下をどうぞ．

- 2025年度「機械学習I」 第4回 https://www-tlab.math.ryukoku.ac.jp/wiki/?ML/2025#ex04
- 2025年度「機械学習I」 第5回 https://www-tlab.math.ryukoku.ac.jp/wiki/?ML/2025#ex05


----
## 準備
----


In [ ]:
# 準備あれこれ
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation, rc  # アニメーションのため
import pandas as pd
import seaborn
seaborn.set_theme()

---
## ロジスティック回帰
---

**ロジスティック回帰** (logistic regression) は「回帰」という名前が付いているが，教師ありの「識別」問題を解くための手法である．


---
### 2クラス識別のロジスティック回帰


$D$ 次元の入力データを2つのクラスのいずれかに分類する問題を考える．入力データは $D$ 次元ベクトル $\pmb{x} = (x_1, x_2, \ldots, x_D)$ で表され，この入力がどちらのクラスに属するかはクラスラベル $y \in \{0, 1\}$ によって表されるものとする．

学習データは，$N$ 個の入力とそれに対応する正解ラベルのペアから構成され，次のように与えられる：

$$
(\pmb{x}_1, y_1), (\pmb{x}_2, y_2), (\pmb{x}_N, y_N)
$$

ここで，$\pmb{x}_n \in R^{D}$ は $n$ 番目の入力データを表し，対応する $y_n \in \{0, 1\}$ はこの入力が属するクラスの正解を表す．

例えば，様々な画像を「犬」と「猫」の2クラスに分類する問題を考える場合，$y_n = 0$ ならば画像 $\pmb{x}_n$ は「犬」のサンプルであり，$y_n = 1$ ならば「猫」のサンプルであるとする（ラベルの割り当て委は逆でも構わない）．




#### モデル

ロジスティック回帰では，$\pmb{x}$ が与えられたときにそれが $y = 1$ に対応するクラスに属する確率 $\Pr(y = 1 | \pmb{x})$ を，次のようにモデル化する．

$$
\begin{aligned}
\Pr(y = 1 | \pmb{x}) = f(\pmb{x}) &= \sigma(w_0 + w_1x_1+\cdots + w_Dx_D) = \sigma\left(w_0 + \sum_{d=1}^{D}w_dx_d \right) \\
&= \frac{1}{1+\exp{\left( - \left( w_0 + \sum_{d=1}^{D}w_dx_d \right) \right)}} \qquad (1)
\end{aligned}
$$

パラメータは，$w_0, w_1, \ldots, w_D$ の $(D+1)$ 個ある．



関数 $\sigma(s)$ は **シグモイド関数** （ロジスティックシグモイド関数, sigmoid function, logistic sigmoid function）と呼ばれるものであり，次のように定義される．

$$
\sigma(s) = \frac{1}{1+\exp{(-s)}}\qquad (2)
$$

式から明らかなように，任意の実数 $s$ に対して $ 0 < \sigma(s) < 1$ となる．
また，$\Pr(y = 0 | \pmb{x}) = 1 - f(\pmb{x})$ とする．

In [ ]:
# シグモイド関数の値を計算
xmin, xmax = -6, 6
X = np.linspace(xmin, xmax, num=100)
Y = 1/(1+np.exp(-X))

# グラフに描く
fig = plt.figure(facecolor='white')
ax = fig.add_subplot(111)
ax.set_xlim(xmin, xmax)
ax.set_ylim(-0.1, 1.1)
ax.axhline(y=0, color='black', linestyle='-')
ax.axvline(x=0, color='black', linestyle='-')
ax.axhline(y=1, color='gray', linestyle='--')
ax.plot(X, Y, linewidth=2)
ax.set_xlabel('$s$')
ax.set_ylabel('$\sigma(s)$')
#ax.legend()
plt.show()

#### モデルの尤度と交差エントロピー

上記のように定義したロジスティック回帰モデルを用いると，ある入力 $\pmb{x}_n$ を与えたときにそれがクラス $y_n$ に属する確率 $\Pr(y_n|\pmb{x}_n)$ は

$$
f(\pmb{x}_n)^{y_n} \cdot (1 - f(\pmb{x}_n))^{1-y_n}
$$

と表される．したがって，この値が，一つのデータ $(\pmb{x}_n, y_n)$ に対するこのモデルの尤度 (likelihood) を与える．

さらに，個々のデータが独立かつ同一の確率分布に従っている（i.i.d. である（注1））と仮定すると，学習データ全体に対するモデルの尤度 $L$ は次の式で与えられる．

$$
L = \prod_{n=1}^{N} f(\pmb{x}_n)^{y_n} \cdot (1 - f(\pmb{x}_n))^{1-y_n}
$$

ロジスティック回帰では，この尤度 $L$ を最大化するモデルパラメータ $w_0, w_1, \ldots, w_D$ を推定する．これは，「最尤推定」と呼ばれる手法の一種である．

ただし，実際の計算においては，尤度そのものは扱いづらいので，その対数をとった対数尤度(log-likelihood)

$$
\log{L} = \sum_{n=1}^{N} ( y_n \log{f(\pmb{x}_n)} + (1 - y_n) \log{(1 - f(\pmb{x}_n))} )
$$

の最大化，あるいは，対数尤度に負号を付けた **交差エントロピー** （cross entropy，注2）

$$
H = -\log{L} = - \sum_{n=1}^{N} ( y_n \log{f(\pmb{x}_n)} + (1 - y_n) \log{(1 - f(\pmb{x}_n))} ) \qquad (3)
$$

の最小化の問題として扱うことが多い．

<br>
<hr width="50%" align="left">
<span style="font-size: 75%">
※注1: i.i.d. は independent and identically distributed の略．<br>
※注2: 交差エントロピーは，情報理論において，モデルの予測と真の分布との差異を測る指標とされるものである．
</span>

#### 勾配降下法による学習

線形回帰モデルの場合，モデルの出力と正解の値との間の二乗誤差 $E$ を最小化するパラメータ $\pmb{w}$ は， $\frac{\partial E}{\partial \pmb{w}} = \pmb{0}$ とおいて得られる連立方程式を解くことで（一撃の計算で）求まっていた．しかし，ロジスティック回帰モデルの尤度最大化/交差エントロピー最小化の場合は，そのように簡単には解が求まらない．パラメータの初期値を適当に定めて，その値を修正して徐々に目的関数を最小化していく，逐次最適化を行う必要がある．

ロジスティック回帰モデルの学習に用いることのできる最適化手法はいくつかあるが，ここでは **勾配降下法** (gradient descent method) による交差エントロピーの最小化の方法を説明する．勾配降下法は，最小化したい目的関数のモデルパラメータに関する「勾配」を計算し，目的関数の値が小さくなる方向へ（勾配の向きと逆の方向へ）パラメータを微修正することを繰り返す最適化アルゴリズムである．

ロジスティック回帰モデルの学習に勾配降下法を適用して交差エントロピー $H$ を最小化する場合，モデルパラメータ $\pmb{w}$ に適当な初期値を設定し，その値を次式によって更新することを繰り返す．

$$
\pmb{w}^{\textrm{new}} = \pmb{w} - \eta \nabla{H}(\pmb{w}) \qquad (4)
$$

ここで，$\nabla{H}(\pmb{w})$ は $H$ のパラメータ $\pmb{w}$ に関する勾配，すなわち

$$
\nabla{H}(\pmb{w}) = \left( \frac{\partial H}{\partial w_0}, \frac{\partial H}{\partial w_1}, \ldots, \frac{\partial H}{\partial w_D}\right)
$$

である．また，定数 $\eta > 0$ は学習率（learning rate，学習定数や学習係数とも）と呼ばれ，1回の更新のステップ幅を決める．




勾配降下法によるパラメータ更新式を具体的に求めるため，$\frac{\partial H}{\partial w_d}$ ($d = 0,1,\ldots,D$) を計算しよう．

$$
\ell_n = y_n\log f(\pmb{x}_n) + (1-y_n)\log(1-f(\pmb{x}_n))
$$

とおくと，

$$
\frac{\partial H}{\partial w_d} = -\sum_{n=1}^{N}\frac{\partial \ell_n}{\partial w_d}
$$

と表せるので，$\frac{\partial \ell_n}{\partial w_d}$ を求めればよい．


（この部分は，板書 + 演習の形で説明します）

整理すると，

$$
\frac{\partial \ell_n}{\partial w_d} = (y_n - f(\pmb{x}_n)) x_{n,d}
$$

が得られる（$x_0 \equiv 1$ とおいた）ので，

$$
\frac{\partial H}{\partial w_d} = -\sum_{n=1}^{N}(y_n - f(\pmb{x}_n)) x_{n,d} = \sum_{n=1}^{N}(f(\pmb{x}_n) - y_n) x_{n,d} \qquad (5)
$$

となる．したがって，ベクトルとしてまとめると，式(4)は次のように書ける．

$$
\pmb{w}^{\textrm{new}} = \pmb{w} - \eta \sum_{n=1}^{N}(f(\pmb{x}_n) - y_n) \pmb{x}_n
$$

#### デモ: 2次元2クラス識別問題

ロジスティック回帰を2次元のデータを2クラスに識別する問題に適用してみよう．

In [ ]:
## 2次元正規分布で2クラスのデータを生成する関数

def getData(seed=None):

    if seed != None:
        np.random.seed( seed )

    # two 2-D spherical Gaussians
    X0 = 1.0*np.random.randn(200, 2) + [3.0, 3.0]
    X1 = 1.0*np.random.randn(200, 2) + [7.0, 6.0]
    X  = np.vstack((X0, X1))
    lab0 = np.zeros(X0.shape[0], dtype=int)
    lab1 = np.zeros(X1.shape[0], dtype=int) + 1
    label = np.hstack((lab0, lab1))

    return X, label

# データの準備
X, lab = getData(seed=0)
N, D = X.shape
Y = lab
X = np.vstack((np.ones(N), X.T)).T
print(f'データ数 N = {N}, 次元数 D = {D}')

In [ ]:
# モデル出力の計算
def model(w, X):
    return 1.0 / (1.0 + np.exp(-(X @ w)))

# 交差エントロピーと正解数
def score(Y, Z):
    ce = -np.sum(Y*np.log(Z)+(1.0-Y)*np.log(1.0-Z)) # 交差エントロピー
    count = np.sum((Z >= 0.5)*Y) + np.sum((Z < 0.5)*(1 - Y)) # 正解数
    return ce, count

# 勾配の計算
def grad(X, Y, Z):
    return (Z - Y) @ X

In [ ]:
# パラメータの初期化
w = (np.random.random(D+1) - 0.5) * 0.2 # [-0.1, 0.1) の一様乱数

# 学習率と学習繰り返し回数
eta = 0.2/N
nitr = 1000

fig = plt.figure(facecolor='white', figsize=(10, 5))
ax1 = fig.add_subplot(121, projection='3d')
ax2 = fig.add_subplot(122)
elevation = 20
azimuth = -70
ax1.view_init(elevation, azimuth)
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 10)
ax1.set_zlim(0, 1)
ax1.scatter(X[Y==0, 1], X[Y==0, 2], 0)
ax1.scatter(X[Y==1, 1], X[Y==1, 2], 1)
#fig.show()

ax2.set_xlim(0, nitr)
ax2.set_ylim(0, 300)

aList = []
xx, yy = np.meshgrid(np.linspace(0, 10, num=16), np.linspace(0, 10, num=16))
xxr, yyr = xx.ravel(), yy.ravel()
XX = np.vstack((np.ones(xxr.shape[0]), xxr, yyr)).T

iList = []
ceList = []

for i in range(nitr+1):

    Z = model(w, X)     # モデル出力の計算
    ce, count = score(Y, Z) # 交差エントロピーと正解数の計算
    dw = grad(X, Y, Z) # 勾配の計算
    w -= eta * dw       # パラメータの更新

    if (i < 100 and i % 10 == 0) or i % 100 == 0:
        iList.append(i)
        ceList.append(ce)
        ZZ = model(w, XX)
        zz = ZZ.reshape(xx.shape)
        a1 = ax1.plot_wireframe(xx, yy, zz, color='green')
        a2 = ax2.plot(iList, ceList, color='blue', marker='.')
        rr = count/N*100
        s = f'H = {ce:.3f}\nacc = {rr:.1f}%'
        a3 = ax2.text(500, 240, s, size=20)
        aList.append([a1]+a2 + [a3])

anim = animation.ArtistAnimation(fig, aList, interval=300)
rc('animation', html='jshtml')
plt.close()
anim


---
### 多クラス識別のロジスティック回帰





$D$次元のデータを $K$ 個のクラス $C_1, C_2, \ldots, C_K$ のいずれかに識別する問題を考える．入力データは $D$ 次元ベクトル $\pmb{x} = (x_1, x_2, \ldots, x_D)$ で表され，この入力がどのクラスに属するかはクラスラベル $y \in \{1, 2, \ldots, K\}$ によって表されるものとする．

学習データは，$N$ 個の入力とそれに対応する正解ラベルのペアから構成され，次のように与えられる：

$$
(\pmb{x}_1, y_1), (\pmb{x}_2, y_2), (\pmb{x}_N, y_N)
$$

ここで，$\pmb{x}_n \in R^{D}$ は $n$ 番目の入力データを表し，対応する $y_n \in \{1, 2, \ldots, K\}$ はこの入力が属するクラスの正解を表す．




#### モデル

$K$クラス識別のロジスティック回帰では，$\pmb{x}$ が与えられたときにそれがクラス $C_k$ に属する確率 $\Pr(y = k | \pmb{x})$ を，次のようにモデル化する．


$$
\begin{aligned}
\Pr(y = k | \pmb{x}) = \widehat{z}_k &= \frac{\exp s_k}{\displaystyle\sum_{j=1}^{K}\exp{s_j}} \qquad (k = 1, 2, \ldots, K) \\
s_k &= w_{k,0} + \sum_{d=1}^{D}w_{k,d}x_d
\end{aligned}
$$

このモデルのパラメータは $w_{k,d}$ ($k = 1, 2, \ldots, K, d = 0, 1, \ldots, D$) の $K\times (D+1)$ 個ある．

このモデルでは，$K$ 個のクラスごとに計算された値 $s_k$ をもとに，$\widehat{z}_k = \frac{\exp s_k}{\sum_{j=1}^{K}\exp{s_j}}$ という式で確率 $\Pr(y=k|\pmb{x})$ を定義している($k = 1, 2, \ldots, K$)．この操作は **ソフトマックス関数** (**softmax function**) と呼ばれる．ソフトマックス関数は，任意の $K$ 個の実数 $s_1, s_2, \ldots, s_K$ を，0 から 1の範囲の値に変換し，かつ全体の和が 1 になるようにする．すなわち，$0 < \widehat{z}_k < 1$ かつ $\sum_{k=1}^{K}\widehat{z}_k = 1$ が成り立つ．この性質により，$\widehat{z}_k$ を「入力 $\pmb{x}$ がクラス $C_k$ に属する確率」と解釈することができる．

---

このとき，モデルの出力と正解の値との間の「遠さ」を，次式の交差エントロピーで定義する．
$$
\begin{aligned}
H &= -\sum_{n=1}^{N} \sum_{k=1}^{K} z_{n,k}\log{\widehat{z}_{n,k}}
\end{aligned}
$$
この $H$ の値がなるべく小さくなるようにパラメータ $\{ w_{k,d} \}$ を求めたい．

#### モデルの尤度と交差エントロピー

2クラスの場合と同様の議論を経て，学習データ全体に対するモデルの尤度 $L$ は次式で与えられる．

$$
L = \prod_{n=1}^{N} \prod_{k=1}^{K} \widehat{z}_{n,k}^{z_{n,k}}
$$

ただし，$z_{n,k}$ は，$y_n = k$ である（$n$ 番目の学習データの所属クラスの正解が $k$ 番目のクラスである）ときに $1$ をとり，それ以外のときに $0$ をとるものとする．$K$ クラス識別のロジスティック回帰では，この尤度を最大化するモデルパラメータを推定する．交差エントロピーは次のように与えられる．

$$
\begin{aligned}
H = -\log{L} &= -\sum_{n=1}^{N}\sum_{k=1}^{K} z_{n,k} \log{\widehat{z}_{n,k}}
\end{aligned}
$$

#### 勾配降下法による学習

導出過程は省略するが，上記の交差エントロピーのパラメータに関する勾配は次のようになる．

$$
\frac{\partial H}{\partial w_{k,d}}  = \sum_{n=1}^{N}(\widehat{z}_{n,k} - z_{n,k}) x_{n,d} \qquad(k = 1, 2, \ldots, K, d = 0, 1, \ldots, D)
$$

2クラス識別の場合（式(5)）と見比べると，添字は増えているが同じような形をしていることが分かる．パラメータ更新の式の説明は省略する．

#### デモ: ロジスティック回帰による手書き数字の識別

手書き数字画像を 0 から 9 の10クラスに識別する問題に適用してみよう．

In [ ]:
# 手書き数字データの入手
! wget -nc https://www-tlab.math.ryukoku.ac.jp/~takataka/course/ML/minimnist.npz
rv = np.load('minimnist.npz')
datL = rv['datL'].astype(float)
labL = rv['labL']
datT = rv['datT'].astype(float)
labT = rv['labT']
print(datL.shape, labL.shape, datT.shape, labT.shape)

K = 10 # クラス数

# 学習データの用意
NL, D = datL.shape # 学習データの数と次元数
XL = np.empty((NL, D+1))
XL[:, 0] = 1.0
XL[:, 1:] = datL/255
ZL = np.zeros((NL, K))
for ik in range(K):
    ZL[labL == ik, ik] = 1.0

# テストデータの用意
NT, _ = datT.shape # テストデータの数
XT = np.empty((NT, D+1))
XT[:, 0] = 1.0
XT[:, 1:] = datT/255
ZT = np.zeros((NT, K))
for ik in range(K):
    ZT[labT == ik, ik] = 1.0

In [ ]:
# パラメータの初期化
W = (np.random.random((K, D+1)) - 0.5) * 0.2 # [-0.1, 0.1) の一様乱数

# 学習率と学習繰り返し回数
eta = 0.01
nitr = 20000

# 学習
for i in range(nitr+1):

    # 学習データの一つをランダムに選択
    n = np.random.randint(NL)
    x, z = XL[n, :], ZL[n]
    # モデル出力の計算
    exps = np.exp(W @ x)
    zt = exps / np.sum(exps)
    # 確率的勾配降下法
    dW = (zt - z)[:, np.newaxis] @ x[np.newaxis, :]
    W -= eta * dW

    if (i < 1000 and i % 100 == 0) or (i % 1000 == 0):
        # モデル出力の計算
        exps = np.exp(XL @ W.T)
        Zt = exps / np.sum(exps, axis=1)[:, np.newaxis]
        # 交差エントロピー
        ce = -np.sum(ZL * np.log(Zt))
        # 正解数
        count = np.sum(labL == np.argmax(Zt, axis=1))
        print(f'{i}  {ce/NL:.3f}  {count/NL:.3f}')

print()

# テスト
exps = np.exp(XT @ W.T)
Zt = exps / np.sum(exps, axis=1)[:, np.newaxis]
ce = -np.sum(ZT * np.log(Zt))
count = np.sum(labT == np.argmax(Zt, axis=1))
print(f'テスト: {ce/NT:.3f}  {count/NT:.3f}')

上記のコードでは，パラメータの最適化に，「確率的勾配降下法」という方法を用いています．どのようなものかについては次回解説の予定です．